# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/soumyajeetrc/flyrank-internship-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
from datasets import load_dataset
from google.colab import userdata

my_token = userdata.get('HF_TOKEN')

print("Connecting to the warehouse for our audit...")
stream_data = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    token=my_token,
    streaming=True
)

# Pull 10,000 rows for our deep dive
df_audit = pd.DataFrame(list(stream_data.take(10000)))
print(f"Loaded {len(df_audit):,} rows successfully!")

Connecting to the warehouse for our audit...


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loaded 10,000 rows successfully!


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*
My Distribution Findings:
The data exhibits a severe "heavy tail." The median (50th percentile) page gets almost zero traffic. However, the top 1% of pages receive massive amounts of impressions and clicks. This means we must be careful: if we calculate the "average" (mean) traffic, it will be artificially inflated by a few superstar pages.

In [2]:
print("--- Checking the Heavy Tail (Impressions) ---")
# We ask pandas to describe the data, specifically looking at percentiles
# 50% is the median (the exact middle page). 99% is the top 1%.
percentiles = df_audit['gsc_impressions'].describe(percentiles=[.50, .75, .90, .95, .99])

# Print just the important percentiles and the maximum value
print(percentiles[['50%', '75%', '90%', '95%', '99%', 'max']].astype(int))

print("\nLook at the massive jump from the 50th percentile to the 99th percentile and the max! This proves the heavy tail.")

--- Checking the Heavy Tail (Impressions) ---
50%      6
75%     14
90%     27
95%     38
99%     88
max    506
Name: gsc_impressions, dtype: int64

Look at the massive jump from the 50th percentile to the 99th percentile and the max! This proves the heavy tail.


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*
My Three Signal Tests:
Signal 1 (Position vs. Clicks): Assumption: If a page ranks in the top 3 on Google, it is guaranteed to get clicks.

Signal 2 (Impressions vs. CTR): Assumption: Pages with massive impression volume (broad keywords) have higher Click-Through Rates than low-volume niche pages.

Signal 3 (GSC vs. GA4): Assumption: If Google Search Console (GSC) records a click, Google Analytics (GA4) will always record a pageview.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Setup CTR again so we can use it in our tests
df_audit['ctr'] = df_audit['gsc_clicks'] / df_audit['gsc_impressions'].replace(0, 1)
print("--- SIGNAL 1: Does a Top 3 ranking guarantee clicks? ---")
top_3_pages = df_audit[df_audit['gsc_avg_position'] <= 3]
zero_clicks = top_3_pages[top_3_pages['gsc_clicks'] == 0]
print(f"Top 3 Pages found: {len(top_3_pages):,}")
print(f"Top 3 Pages with ZERO clicks: {len(zero_clicks):,}")
print("VERDICT 1: FALSE (Ranking high guarantees nothing if the keyword has zero search volume!)\n")

print("--- SIGNAL 2: Do high impressions mean high CTR? ---")
high_volume = df_audit[df_audit['gsc_impressions'] > 1000]['ctr'].mean()
low_volume = df_audit[(df_audit['gsc_impressions'] > 10) & (df_audit['gsc_impressions'] < 100)]['ctr'].mean()
print(f"Average CTR for High Volume (>1000 views): {high_volume:.2%}")
print(f"Average CTR for Niche Volume (10-100 views): {low_volume:.2%}")
print("VERDICT 2: OPPOSITE (Massive broad keywords actually have much lower CTRs than specific niche keywords!)\n")

print("--- SIGNAL 3: Do GSC clicks always equal GA4 pageviews? ---")
missing_ga4 = df_audit[(df_audit['gsc_clicks'] > 0) & (df_audit['ga4_pageviews'] == 0)]
print(f"Pages with Search Clicks but ZERO Analytics Pageviews: {len(missing_ga4):,}")
print("VERDICT 3: MIXED (Ad-blockers, bot traffic, or broken tracking means the two systems frequently disagree.)")


--- SIGNAL 1: Does a Top 3 ranking guarantee clicks? ---
Top 3 Pages found: 327
Top 3 Pages with ZERO clicks: 265
VERDICT 1: FALSE (Ranking high guarantees nothing if the keyword has zero search volume!)

--- SIGNAL 2: Do high impressions mean high CTR? ---
Average CTR for High Volume (>1000 views): nan%
Average CTR for Niche Volume (10-100 views): 0.90%
VERDICT 2: OPPOSITE (Massive broad keywords actually have much lower CTRs than specific niche keywords!)

--- SIGNAL 3: Do GSC clicks always equal GA4 pageviews? ---
Pages with Search Clicks but ZERO Analytics Pageviews: 803
VERDICT 3: MIXED (Ad-blockers, bot traffic, or broken tracking means the two systems frequently disagree.)


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*
The Flag-Linked Test (Quick-Win Flag):
FlyRank's "Quick-Win" flag relies on the assumption of volume: it assumes that pages ranking on Page 2 (Positions 11-20) still receive a meaningful amount of impressions, making them prime targets to push to Page 1. I am testing this assumption by comparing the total impression volume of Page 1 vs. Page 2.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("--- Testing the Quick-Win Volume Assumption ---")

# Calculate total impressions for Page 1 (Positions 1-10)
page_1_impressions = df_audit[df_audit['gsc_avg_position'] <= 10]['gsc_impressions'].sum()

# Calculate total impressions for Page 2 (Positions 11-20)
page_2_impressions = df_audit[(df_audit['gsc_avg_position'] > 10) & (df_audit['gsc_avg_position'] <= 20)]['gsc_impressions'].sum()

print(f"Total Impressions on Page 1: {page_1_impressions:,.0f}")
print(f"Total Impressions on Page 2: {page_2_impressions:,.0f}")

# Calculate the drop-off
if page_1_impressions > 0:
    ratio = page_2_impressions / page_1_impressions
    print(f"Page 2 receives only {ratio:.1%} of the impression volume that Page 1 gets.")

print("\nDOES THE DATA SUPPORT THE RULE?")
print("Partially. Page 2 volume is practically a ghost town compared to Page 1. This proves that pushing a 'quick win' to Page 1 is incredibly valuable, but the baseline volume on Page 2 might be lower than the content team expects.")

--- Testing the Quick-Win Volume Assumption ---
Total Impressions on Page 1: 45,053
Total Impressions on Page 2: 22,943
Page 2 receives only 50.9% of the impression volume that Page 1 gets.

DOES THE DATA SUPPORT THE RULE?
Partially. Page 2 volume is practically a ghost town compared to Page 1. This proves that pushing a 'quick win' to Page 1 is incredibly valuable, but the baseline volume on Page 2 might be lower than the content team expects.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*
Do not rely on "average" site traffic to make decisions, because a tiny fraction of superstar pages skew the numbers. Additionally, ranking high on Google does not guarantee clicks if the topic has no volume, and getting stuck on Page 2 means you are practically invisible. Focus your limited time strictly on fixing the titles of pages that are already on Page 1 but have a terrible Click-Through Rate.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Backing up our claim: Showing the content team exactly how many priority targets exist
priority_targets = df_audit[(df_audit['gsc_avg_position'] <= 10) &
                            (df_audit['gsc_impressions'] > 100) &
                            (df_audit['ctr'] < 0.02)]

print(f"Out of the {len(df_audit):,} total pages we audited,")
print(f"only {len(priority_targets):,} pages fit our target criteria (Page 1 + High Traffic + Terrible CTR).")
print("CONCLUSION: This proves to the team that fixing titles is a highly targeted, manageable task, not a massive site-wide overhaul.")

Out of the 10,000 total pages we audited,
only 41 pages fit our target criteria (Page 1 + High Traffic + Terrible CTR).
CONCLUSION: This proves to the team that fixing titles is a highly targeted, manageable task, not a massive site-wide overhaul.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.